In [11]:
from pathlib import Path
import pickle
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
from pathlib import Path

DATA_DIR = Path("Data")
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Target present: {'Survived' in train_df.columns}")

Train shape: (891, 12)
Test shape: (418, 11)
Target present: True


In [6]:
def make_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create compact, leakage-safe features available in both train and test."""
    features = data.copy()
    features["Title"] = (
        features["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()
    )
    features["Title"] = features["Title"].replace({
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"
    })
    rare_titles = ~features["Title"].isin(["Mr", "Miss", "Mrs", "Master"])
    features.loc[rare_titles, "Title"] = "Rare"
    features["FamilySize"] = features["SibSp"] + features["Parch"] + 1
    features["IsAlone"] = (features["FamilySize"] == 1).astype(int)
    features["FarePerPerson"] = features["Fare"] / features["FamilySize"]
    features["CabinDeck"] = features["Cabin"].fillna("U").str[0]
    features["TicketGroupSize"] = features.groupby("Ticket")["Ticket"].transform("size")
    return features.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

X = make_features(train_df)
y = train_df["Survived"].astype(int)
X_test = make_features(test_df)

print("Model features:", list(X.columns))
print("Missing values before pipeline imputation:", int(X.isna().sum().sum()))
print("Class balance:\n", y.value_counts(normalize=True).rename("proportion"))

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

Model features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'FarePerPerson', 'CabinDeck', 'TicketGroupSize']
Missing values before pipeline imputation: 179
Class balance:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [7]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=2, max_features="sqrt", random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500, min_samples_leaf=2, max_features="sqrt", random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=42
    ),
}

pipelines = {
    name: Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    for name, estimator in models.items()
}

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

Numeric features: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'TicketGroupSize']
Categorical features: ['Sex', 'Embarked', 'Title', 'CabinDeck']


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}

cv_rows = []
for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1, return_train_score=False
    )
    cv_rows.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_f1_mean": scores["test_f1"].mean(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_roc_auc_mean", ascending=False).reset_index(drop=True)
print(cv_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

               model  cv_accuracy_mean  cv_accuracy_std  cv_f1_mean  cv_roc_auc_mean
       Random Forest            0.8343           0.0230      0.7744           0.8886
HistGradientBoosting            0.8287           0.0209      0.7676           0.8878
         Extra Trees            0.8259           0.0243      0.7636           0.8835
 Logistic Regression            0.8245           0.0326      0.7692           0.8661


In [12]:
validation_rows = []
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_valid)
    probabilities = pipeline.predict_proba(X_valid)[:, 1]
    validation_rows.append({
        "model": name,
        "validation_accuracy": accuracy_score(y_valid, predictions),
        "validation_f1": f1_score(y_valid, predictions),
        "validation_roc_auc": roc_auc_score(y_valid, probabilities),
    })

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("validation_roc_auc", ascending=False)
    .reset_index(drop=True)
)
print(validation_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

best_model_name = cv_results.iloc[0]["model"]
best_pipeline = pipelines[best_model_name]
best_pipeline.fit(X, y)

test_predictions = best_pipeline.predict(X_test).astype(int)
prediction_path = Path("titanic_predictions.csv")
pd.DataFrame({"PassengerId": test_df["PassengerId"], "Survived": test_predictions}).to_csv(
    prediction_path, index=False
)

model_path = Path("titanic_best_model.pkl")
with model_path.open("wb") as file:
    pickle.dump(best_pipeline, file)

print(f"Selected model: {best_model_name}")
print(f"Saved model: {model_path.resolve()}")
print(f"Saved predictions: {prediction_path.resolve()}")

               model  validation_accuracy  validation_f1  validation_roc_auc
 Logistic Regression               0.8436         0.7879              0.8727
         Extra Trees               0.7933         0.7087              0.8503
       Random Forest               0.8045         0.7287              0.8424
HistGradientBoosting               0.7877         0.7246              0.8287
Selected model: Random Forest
Saved model: C:\Users\Admin\Documents\GitHub\Titanic-Model\titanic_best_model.pkl
Saved predictions: C:\Users\Admin\Documents\GitHub\Titanic-Model\titanic_predictions.csv


In [13]:
with model_path.open("rb") as file:
    loaded_model = pickle.load(file)

loaded_predictions = loaded_model.predict(X_test)
assert len(loaded_predictions) == len(test_df)
assert hasattr(loaded_model, "predict_proba")

print(f"Round-trip load successful: {best_model_name}")
print(f"Predictions generated after reload: {len(loaded_predictions)}")
print(f"Positive predictions: {int(loaded_predictions.sum())}")

Round-trip load successful: Random Forest
Predictions generated after reload: 418
Positive predictions: 150
